# MLB Model Training And Backtest

                Train candidate MLB winner models from the feature store, hold out 2026 YTD,
                and write predictions/metrics for the performance history hub.

In [ ]:
import json
                import pandas as pd
                from pathlib import Path

                metrics_path = Path("cache/mlb_backtest_metrics_2026_ytd.json")
                preds_path = Path("cache/mlb_backtest_predictions_2026_ytd.csv")

In [ ]:
# Rebuild metrics and predictions.
                # %run ../scripts/backtest_mlb_winners.py --features-path cache/mlb_feature_store_2021_2026.parquet --validation-season 2025 --test-season 2026 --predictions-output cache/mlb_backtest_predictions_2026_ytd.csv --metrics-output cache/mlb_backtest_metrics_2026_ytd.json

In [ ]:
metrics = json.loads(metrics_path.read_text())
                metrics["selected_refit_test"]

In [ ]:
rows = []
                for name, values in metrics["candidate_metrics"].items():
                    row = {"model": name}
                    row.update({f"validation_{k}": v for k, v in values["validation"].items()})
                    row.update({f"test_{k}": v for k, v in values["test"].items()})
                    rows.append(row)
                pd.DataFrame(rows)

In [ ]:
preds = pd.read_csv(preds_path)
                preds.assign(correct=preds.pick_won.astype(bool)).groupby("pick_side").agg(
                    games=("game_pk", "count"),
                    accuracy=("correct", "mean"),
                    avg_home_prob=("home_win_prob", "mean"),
                )